<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 03a — Custom Training & Batch Inference

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h00

</div>

## 📋 Lab Overview

In a production ML workflow, training typically doesn't happen in a notebook — it runs as a **managed job** on cloud infrastructure. Once a model is trained, it often needs to score large volumes of data offline rather than serving individual requests. Vertex AI provides **Custom Training Jobs** for scalable, reproducible training and **Batch Prediction** for high-throughput offline inference.

### Learning Objectives

1. Understand Vertex AI **pre-built containers** and how they simplify managed training.
2. Write a **training script** compatible with Vertex AI's conventions (`AIP_MODEL_DIR`).
3. Configure and submit a **`CustomTrainingJob`** to Vertex AI.
4. Prepare input data in **JSONL format** and upload it to Cloud Storage.
5. Launch a **`BatchPredictionJob`** and retrieve results from GCS.
6. Evaluate batch prediction results programmatically.

### Business Context

Imagine you are deploying an image classification model for an e-commerce platform. Product images arrive in bulk every night and need to be categorized before the catalog update. **Batch inference** is the right pattern here: high throughput, no latency constraint, and cost-efficient because resources are provisioned only for the duration of the job.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install dependencies, imports, GCP configuration |
| 1 | Custom Training Job | Pre-built containers, training script, submit job |
| 2 | Batch Prediction | Prepare JSONL input, launch job, retrieve results |
| 3 | Cleanup | Delete training job and batch prediction resources |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [2]:
%pip install --upgrade --quiet google-cloud-aiplatform google-cloud-storage pillow numpy

### 0.2 Imports

In [3]:
import os
import json
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
from PIL import Image

from google.cloud import aiplatform

print(f"Vertex AI SDK version: {aiplatform.__version__}")

Vertex AI SDK version: 1.140.0


### 0.3 Configuration

Replace the placeholders below with your own GCP project details. These constants will be reused in **Lab 03b** (online endpoints), so keep them consistent.

In [4]:
# ── Constants ──
PROJECT_ID = "isae-sdd-481407"          # @param {type:"string"}
LOCATION = "europe-west3"                # @param {type:"string"}
BUCKET_URI = "gs://bucket-lab03"    # @param {type:"string"}

##############################  TODO  ##############################
# Set your_name to a unique lowercase identifier (e.g. first letter of first name + last name).
your_name = "mregaieg"  # TODO
####################################################################

aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET_URI,
)
print(f"✅ Vertex AI initialized — project={PROJECT_ID}, location={LOCATION}")

✅ Vertex AI initialized — project=isae-sdd-481407, location=europe-west3


---
## 1 · Custom Training Job

In Lab 02b you uploaded a pre-trained model to the Model Registry. Here, you will **train a model from scratch** using Vertex AI's managed training service. This means the training code runs on GCP infrastructure (not your local machine), and the resulting model artifacts are automatically saved to Cloud Storage.

### 1.1 Pre-built containers

Vertex AI offers **pre-built Docker containers** with popular ML frameworks (TensorFlow, PyTorch, scikit-learn, XGBoost). You don't need to write a Dockerfile — just pick a container that matches your framework and Python version, then provide your training script.

> 📖 **Docs:**
> - [Pre-built containers for training](https://cloud.google.com/vertex-ai/docs/training/pre-built-containers)
> - [Pre-built containers for prediction](https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers)

In [5]:
# Training container: TensorFlow 2.17 CPU
TRAIN_IMAGE = "europe-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest"

# Prediction (serving) container: TensorFlow 2.15 CPU
DEPLOY_IMAGE = "europe-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"

print(f"Training image:   {TRAIN_IMAGE}")
print(f"Prediction image: {DEPLOY_IMAGE}")

Training image:   europe-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest
Prediction image: europe-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest


**✏️ Question 1 — Pre-built containers**

a) Why does Vertex AI use *separate* container images for training and prediction? What are the advantages of this separation?

b) When would you use a **custom container** instead of a pre-built one?

---
*Your answer:*

**a)** Vertex AI uses separate containers for training and prediction because they have different purposes. The training container is used to build and train the model, so it may require more libraries and tools. The prediction container is used to serve the trained model, so it is usually lighter and focused on inference. This separation makes maintenance easier, simplifies deployment, and helps avoid unnecessary dependencies in the production environment.

**b)** A custom container is used when the pre-built containers do not meet the project requirements. For example, a project may need a specific framework version, additional libraries, or custom preprocessing or prediction logic. In that case, a custom container provides more flexibility and control over the environment.



---

### 1.2 The training script

The cell below writes a `task.py` file that defines and trains a CNN on the CIFAR-10 dataset. This script is designed to run **inside the pre-built container** on Vertex AI — not in your notebook.

Key conventions to notice:
- **`AIP_MODEL_DIR`**: an environment variable automatically set by Vertex AI. Your script must save the trained model to this path so Vertex AI can register it.
- **`argparse`**: training parameters (epochs, learning rate, etc.) are passed as command-line arguments. This makes the script configurable from the job submission.
- **Distribution strategies**: TensorFlow's `tf.distribute` API lets the same code run on 1 GPU, multiple GPUs, or multiple machines.

Read the script carefully — you will need to understand it for the questions that follow.

In [6]:
%%writefile task.py
# Training script for CIFAR-10 image classification
# This script runs INSIDE a Vertex AI pre-built container.

import tensorflow_datasets as tfds
import tensorflow as tf
from tensorflow.python.client import device_lib
import argparse
import os
import sys

tfds.disable_progress_bar()

parser = argparse.ArgumentParser()
parser.add_argument('--lr', dest='lr', default=0.01, type=float,
                    help='Learning rate.')
parser.add_argument('--epochs', dest='epochs', default=10, type=int,
                    help='Number of epochs.')
parser.add_argument('--steps', dest='steps', default=200, type=int,
                    help='Number of steps per epoch.')
parser.add_argument('--distribute', dest='distribute', type=str,
                    default='single', help='Distribution strategy.')
args = parser.parse_args()

print(f"Python Version = {sys.version}")
print(f"TensorFlow Version = {tf.__version__}")
print(f"TF_CONFIG = {os.environ.get('TF_CONFIG', 'Not found')}")

# ── Distribution strategy ──
if args.distribute == 'single':
    if tf.test.is_gpu_available():
        strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    else:
        strategy = tf.distribute.OneDeviceStrategy(device="/cpu:0")
elif args.distribute == 'mirror':
    strategy = tf.distribute.MirroredStrategy()
elif args.distribute == 'multi':
    strategy = tf.distribute.experimental.MultiWorkerMirroredStrategy()

print(f"num_replicas_in_sync = {strategy.num_replicas_in_sync}")

# ── Dataset ──
BUFFER_SIZE = 10000
BATCH_SIZE = 64

def make_datasets_unbatched():
    def scale(image, label):
        image = tf.cast(image, tf.float32)
        image /= 255.0
        return image, label
    datasets, _ = tfds.load(name='cifar10', with_info=True, as_supervised=True)
    return datasets['train'].map(scale).cache().shuffle(BUFFER_SIZE).repeat()

# ── Model ──
def build_and_compile_cnn_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])
    model.compile(
        loss=tf.keras.losses.sparse_categorical_crossentropy,
        optimizer=tf.keras.optimizers.SGD(learning_rate=args.lr),
        metrics=['accuracy'],
    )
    return model

# ── Training ──
NUM_WORKERS = strategy.num_replicas_in_sync
GLOBAL_BATCH_SIZE = BATCH_SIZE * NUM_WORKERS
MODEL_DIR = os.getenv("AIP_MODEL_DIR")

train_dataset = make_datasets_unbatched().batch(GLOBAL_BATCH_SIZE)

with strategy.scope():
    model = build_and_compile_cnn_model()

model.fit(x=train_dataset, epochs=args.epochs, steps_per_epoch=args.steps)
model.export(MODEL_DIR)

Writing task.py


**✏️ Question 2 — Training script conventions**

a) What is the role of the `AIP_MODEL_DIR` environment variable? What would happen if the script saved the model to a hardcoded local path instead?

b) The script supports three distribution strategies: `single`, `mirror`, and `multi`. In which scenario would you choose `mirror` over `single`?

---
*Your answer:*

**a)** `AIP_MODEL_DIR` tells the training script where to save the trained model so that Vertex AI can store it and use it later for deployment. If the model was saved to a hardcoded local path, it would only stay on that machine, and Vertex AI might not be able to find or upload it correctly after training.

**b)** The `mirror` strategy is chosen when training runs on a single machine with multiple GPUs. It allows the model to use all available GPUs at the same time, which can make training faster. The `single` strategy is more suitable when only one CPU or one GPU is being used.


---

### 1.3 Configure training arguments

Before submitting the job, we define the hyperparameters and the distribution strategy to use. These values are passed to `task.py` as command-line arguments.

In [7]:
JOB_NAME = "cifar10-custom-training"
MODEL_DIR = f"{BUCKET_URI}/{your_name}/{JOB_NAME}"

EPOCHS = 20
STEPS = 100
TRAIN_STRATEGY = "single"

CMDARGS = [
    f"--epochs={EPOCHS}",
    f"--steps={STEPS}",
    f"--distribute={TRAIN_STRATEGY}",
]

print(f"Job name:  {JOB_NAME}")
print(f"Model dir: {MODEL_DIR}")
print(f"Args:      {CMDARGS}")

Job name:  cifar10-custom-training
Model dir: gs://bucket-lab03/mregaieg/cifar10-custom-training
Args:      ['--epochs=20', '--steps=100', '--distribute=single']


### 1.4 Submit the Custom Training Job

Use `CustomTrainingJob` to package and submit the training script. The `run()` method starts the job and blocks until it completes.

> 📖 **Docs:**
> - [`CustomTrainingJob`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.CustomTrainingJob)
> - [`CustomTrainingJob.run()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.CustomTrainingJob#google_cloud_aiplatform_CustomTrainingJob_run)

In [8]:
##############################  TODO  ##############################
# Submit a Custom Training Job for the task.py script.
#
# 1. Instantiate a CustomTrainingJob that packages task.py inside
#    the correct pre-built container. Don't forget to declare the
#    pip dependency needed by the training script (check the first import
#    in task.py) and to specify which serving container the resulting
#    model should use.
#
# 2. Launch the job on a single n1-standard-4 machine with the args.
#    Make sure the model artifacts are written to MODEL_DIR.

job = aiplatform.CustomTrainingJob(
    display_name=JOB_NAME,                          # TODO
    script_path="task.py",                           # TODO
    container_uri=TRAIN_IMAGE,                         # TODO
    requirements=["tensorflow-datasets"],                          # TODO
    model_serving_container_image_uri=DEPLOY_IMAGE,     # TODO
)

MODEL_DISPLAY_NAME = "cifar10-model"

model = job.run(
    model_display_name=MODEL_DISPLAY_NAME,   # TODO
    args=CMDARGS,                 # TODO
    replica_count=1,        # TODO
    machine_type="n1-standard-4",         # TODO
    base_output_dir=MODEL_DIR,      # TODO
)
####################################################################

> ⏳ **Training takes ~10–15 minutes.** While you wait, open the [Vertex AI Training console](https://console.cloud.google.com/vertex-ai/training/training-pipelines) in your browser and monitor the job status.

> 💡 **Tip:** Notice how the console shows the training logs streamed from the container. This is the same output you would see if running `task.py` locally.

In [9]:
# Verify the model was created
print(f"   Model resource name: {model.resource_name}")
print(f"   Model display name:  {model.display_name}")
print(f"   Model URI:           {model.uri}")

   Model resource name: projects/588027604890/locations/europe-west3/models/3825231739358281728
   Model display name:  cifar10-model
   Model URI:           gs://bucket-lab03/mregaieg/cifar10-custom-training/model


**✏️ Question 3 — Custom Training Jobs**

a) In the GCP console, find the training job you just submitted. What **machine type** was used? How much did this job cost approximately? (Check the [pricing page](https://cloud.google.com/vertex-ai/pricing#custom-trained_models).)

b) If you needed to train a much larger model (e.g., a ResNet-50) on the full CIFAR-10 dataset, what changes would you make to the job configuration? Think about `machine_type`, `accelerator_type`, and `distribute`.

---
*Your answer:*

**a)** The training job used the machine type **n1-standard-4**.  
In the region **europe-west3**, the price is **$0.2815085 per hour**.  
The job ran for **5 minutes 8 seconds**, which is **308 seconds**, or **0.08556 hours**.

Cost calculation:  
0.2815085 × 0.08556 ≈ **$0.02408**

So the approximate cost of the training job is **about $0.024 (around 2.4 cents)**.

**b)** To train a much larger model such as **ResNet-50** on the full CIFAR-10 dataset, the configuration should use stronger resources. A larger `machine_type` (for example `n1-standard-8` or `n1-standard-16`) would provide more CPU and memory. Adding a GPU with `accelerator_type` would significantly speed up deep learning training. The `distribute` strategy could also be changed to `mirror` to use multiple GPUs on one machine, or `multi` to distribute training across multiple machines.


---

---
## 2 · Batch Prediction

Now that the model is trained and registered, we can use it for **batch inference**. Batch prediction is ideal when you need to score a large dataset asynchronously — for instance, classifying thousands of images overnight.

Vertex AI manages the compute infrastructure: it spins up prediction nodes, distributes the input data, collects results, and writes them to Cloud Storage.

### 2.1 Download test data

We use a small set of CIFAR-10 test images hosted in a public GCS bucket. The file naming convention encodes the true label: `image_{label}_{index}.jpg`.

In [10]:
# Download test images from public GCS bucket
! gsutil -m cp -r gs://cloud-samples-data/ai-platform-unified/cifar_test_images .

print("✅ Test images downloaded.")
! ls cifar_test_images/

Copying gs://cloud-samples-data/ai-platform-unified/cifar_test_images/image_2_9.jpg...
Copying gs://cloud-samples-data/ai-platform-unified/cifar_test_images/image_3_7.jpg...
Copying gs://cloud-samples-data/ai-platform-unified/cifar_test_images/image_4_10.jpg...
Copying gs://cloud-samples-data/ai-platform-unified/cifar_test_images/image_0_3.jpg...
Copying gs://cloud-samples-data/ai-platform-unified/cifar_test_images/image_0_6.jpg...
Copying gs://cloud-samples-data/ai-platform-unified/cifar_test_images/image_5_4.jpg...
Copying gs://cloud-samples-data/ai-platform-unified/cifar_test_images/image_2_1.jpg...
Copying gs://cloud-samples-data/ai-platform-unified/cifar_test_images/image_7_2.jpg...
Copying gs://cloud-samples-data/ai-platform-unified/cifar_test_images/image_2_8.jpg...
Copying gs://cloud-samples-data/ai-platform-unified/cifar_test_images/image_2_5.jpg...
- [10/10 files][  8.7 KiB/  8.7 KiB] 100% Done                                  
Operation completed over 10 objects/8.7 KiB.    

### 2.2 Load and preprocess test images

The images need to match the format expected by the model: normalized float32 values in [0, 1].

> 📖 **Docs:** [Batch prediction input formats](https://cloud.google.com/vertex-ai/docs/predictions/batch-predictions#batch_request_input), [`os.listdir`](https://docs.python.org/3/library/os.html#os.listdir), [`Image.open`](https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.open)

In [11]:
##############################  TODO  ##############################
# Load test images, normalize them, and extract labels.
#
# 1. List all .jpg files in the IMAGE_DIRECTORY
# 2. Load each image as a numpy array using PIL
# 3. Normalize pixel values to [0, 1] and convert to float32 lists
# 4. Extract the true label from the filename (e.g., "image_2_9.jpg" → label 2)

IMAGE_DIRECTORY = "cifar_test_images"

image_files = [f for f in os.listdir(IMAGE_DIRECTORY) if f.endswith(".jpg")]  # TODO: list .jpg files
image_data = [np.array(Image.open(os.path.join(IMAGE_DIRECTORY, f))) for f in image_files]  # TODO: open the images and convert them as numpy arrays
x_test = [(img.astype(np.float32) / 255.0).tolist() for img in image_data]       # TODO: normalize to [0,1], convert to float32 (numpy), then .tolist()
y_test = [int(f.split("_")[1]) for f in image_files]       # TODO: extract labels from filenames
####################################################################

print(f"   Loaded {len(x_test)} images")
print(f"   Image shape: {np.array(image_data[0]).shape}")
print(f"   Labels: {y_test}")

   Loaded 10 images
   Image shape: (32, 32, 3)
   Labels: [7, 0, 2, 3, 2, 4, 2, 2, 0, 5]


### 2.3 Prepare JSONL input

Vertex AI Batch Prediction accepts several input formats. We use **JSONL** (JSON Lines): one JSON object per line, where each object is a single input instance. The file is uploaded to Cloud Storage so the batch job can read it.

> 📖 **Docs:** [`json.dumps`](https://docs.python.org/3/library/json.html#json.dumps)

In [12]:
##############################  TODO  ##############################
# Write each element of x_test as a JSON line
# Don't forget to add a newline character

BATCH_PREDICTION_INSTANCES_FILE = "batch_prediction_instances.jsonl"
BATCH_PREDICTION_GCS_SOURCE = (
    f"{BUCKET_URI}/{your_name}/batch_prediction_instances/{BATCH_PREDICTION_INSTANCES_FILE}"
)

# Write JSONL locally
with open(BATCH_PREDICTION_INSTANCES_FILE, "w") as f:
    for x in x_test:
        f.write(json.dumps(x)+"\n")  # TODO: write one JSON line per instance
####################################################################

# Upload to GCS
! gsutil cp $BATCH_PREDICTION_INSTANCES_FILE $BATCH_PREDICTION_GCS_SOURCE

print(f"✅ Uploaded instances to: {BATCH_PREDICTION_GCS_SOURCE}")

Copying file://batch_prediction_instances.jsonl [Content-Type=application/octet-stream]...
/ [1 files][617.6 KiB/617.6 KiB]                                                
Operation completed over 1 objects/617.6 KiB.                                    
✅ Uploaded instances to: gs://bucket-lab03/mregaieg/batch_prediction_instances/batch_prediction_instances.jsonl


### 2.4 Launch batch prediction

Now we call `model.batch_predict()` to start the batch prediction job. Vertex AI provisions compute nodes, loads the model, scores all instances, and writes results to GCS.

> 📖 **Docs:** [`Model.batch_predict()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_batch_predict)

In [13]:
##############################  TODO  ##############################
# Launch a batch prediction job.
# Fill in the missing parameters:
#   - instances_format and predictions_format: "jsonl"
#   - job_display_name: a descriptive name
#   - gcs_source: path to the JSONL file on GCS
#   - gcs_destination_prefix: where to write results on GCS
#   - machine_type: "n1-standard-4"

DESTINATION_FOLDER = "batch_prediction_results"
BATCH_PREDICTION_GCS_DEST_PREFIX = f"{BUCKET_URI}/{your_name}/{DESTINATION_FOLDER}"

batch_prediction_job = model.batch_predict(
    instances_format="jsonl",                           # TODO
    predictions_format="jsonl",                         # TODO
    job_display_name="batch-prediction-lab03a-mohamed",                           # TODO
    gcs_source=BATCH_PREDICTION_GCS_SOURCE,                                 # TODO
    gcs_destination_prefix=BATCH_PREDICTION_GCS_DEST_PREFIX,                     # TODO
    machine_type="n1-standard-4",                               # TODO
    starting_replica_count=1,
    max_replica_count=1,
    sync=True,
)
####################################################################

> ⏳ **Batch prediction takes ~10–15 minutes.** You can monitor it in the [Batch Predictions console](https://console.cloud.google.com/vertex-ai/batch-predictions).

### 2.5 Retrieve and evaluate results

Once the job completes, results are written to GCS as JSONL files. Each line contains the model's prediction for one input instance.

In [14]:
# Download results from GCS
RESULTS_DIRECTORY = "prediction_results"
RESULTS_DIRECTORY_FULL = RESULTS_DIRECTORY + "/" + DESTINATION_FOLDER
os.makedirs(RESULTS_DIRECTORY, exist_ok=True)

! gsutil -m cp -r $BATCH_PREDICTION_GCS_DEST_PREFIX $RESULTS_DIRECTORY

print("✅ Results downloaded.")

Copying gs://bucket-lab03/mregaieg/batch_prediction_results/prediction-cifar10-model-2026_03_10T02_16_22_667Z/prediction.errors_stats-00000-of-00001...
Copying gs://bucket-lab03/mregaieg/batch_prediction_results/prediction-cifar10-model-2026_03_10T02_16_22_667Z/prediction.results-00000-of-00001...
/ [2/2 files][619.2 KiB/619.2 KiB] 100% Done                                    
Operation completed over 2 objects/619.2 KiB.                                    
✅ Results downloaded.


In [15]:
# Parse the prediction results
latest_directory = max(
    (os.path.join(RESULTS_DIRECTORY_FULL, d)
     for d in os.listdir(RESULTS_DIRECTORY_FULL)),
    key=os.path.getmtime,
)

results_files = []
for dirpath, subdirs, files in os.walk(latest_directory):
    for file in files:
        if file.startswith("prediction.results"):
            results_files.append(os.path.join(dirpath, file))

results = []
for results_file in results_files:
    with open(results_file, "r") as file:
        results.extend([json.loads(line) for line in file.readlines()])

print(f"✅ Loaded {len(results)} prediction results")

✅ Loaded 10 prediction results


In [17]:
##############################  TODO  ##############################
# Evaluate the batch prediction results.
# 1. Extract predicted labels using np.argmax on each prediction vector
# 2. Compare with y_test to compute accuracy

y_predicted = [np.argmax(result) for result in results]  # TODO: list of predicted labels (use np.argmax)

correct = np.sum(np.array(y_predicted) == np.array(y_test))      # TODO: count correct predictions
total = len(results)        # TODO: total predictions
accuracy = correct/total     # TODO: accuracy = correct / total
####################################################################

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {accuracy:.2%}")

Correct: 2 / 10
Accuracy: 20.00%


**✏️ Question 4 — Batch prediction**

a) What input formats does Vertex AI Batch Prediction support besides JSONL? In which situation would you use BigQuery as input/output instead of GCS?

b) The batch job ran with `max_replica_count=1`. If you had 100,000 images to score, what parameter would you change, and how does Vertex AI distribute the workload across replicas?

---
*Your answer:*

**a)** Besides JSONL, Vertex AI Batch Prediction also supports **CSV files and BigQuery tables** as input formats. BigQuery would be used instead of GCS when the data is already stored in a BigQuery table or when working with large structured datasets that are easier to query and manage directly in BigQuery.

**b)** The parameter to change would be **`max_replica_count`**. Increasing this value allows Vertex AI to use multiple replicas to process the predictions. Vertex AI automatically splits the input data into smaller shards and distributes them across the replicas, so each replica processes a portion of the data in parallel.

---

---
## 3 · Cleanup

Always clean up cloud resources after a lab to avoid unnecessary charges.

> ⚠️ **Important:** Do **not** delete the model if you plan to continue with **Lab 03b** (online endpoints). Only delete the training job and batch prediction job.

In [18]:
# Delete the training job (does not delete the model)
try:
    job.delete()
    print("✅ Training job deleted.")
except Exception as e:
    print(f"Training job cleanup: {e}")

✅ Training job deleted.


In [19]:
# Delete the batch prediction job
try:
    batch_prediction_job.delete()
    print("✅ Batch prediction job deleted.")
except Exception as e:
    print(f"Batch prediction cleanup: {e}")

✅ Batch prediction job deleted.


In [20]:
# Print model info for Lab 06
print("═" * 60)
print("Keep these values for Lab 06:")
print(f"  MODEL_DISPLAY_NAME = \"{MODEL_DISPLAY_NAME}\"")
print(f"  model.resource_name = \"{model.resource_name}\"")
print("═" * 60)

════════════════════════════════════════════════════════════
Keep these values for Lab 06:
  MODEL_DISPLAY_NAME = "cifar10-model"
  model.resource_name = "projects/588027604890/locations/europe-west3/models/3825231739358281728"
════════════════════════════════════════════════════════════


---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| Setup | Configured GCP project and pre-built containers | `aiplatform.init()`, container URIs |
| Custom Training | Submitted a training script as a managed job | `CustomTrainingJob`, `job.run()` |
| Batch Prediction | Scored test data offline via a batch job | `model.batch_predict()`, JSONL, GCS |
| Cleanup | Deleted training and batch prediction resources | `job.delete()` |

**Next lab:** In **Lab 03b**, you will deploy the same model behind an **online endpoint** for real-time predictions, configure **traffic splitting** for A/B testing, wrap the endpoint with **FastAPI**, and run **load tests** with Locust.